In [1]:
import pandas as pd

In [ ]:
df_2015 = pd.read_csv('2015_코드북_복지사회참여문화와여가소득과소비노동.csv')
df_2017 = pd.read_csv('2017_코드북_복지사회참여문화와여가소득과소비노동.csv')
df_2019 = pd.read_csv('2019_코드북_복지사회참여문화와여가소득과소비노동.csv')
df_2019

,항목명,코드,코드의미 및 단위
0,가구일련번호,NaN,NaN
1,가구원일련번호,NaN,NaN
2,만나이,NaN,NaN
3,성별,NaN,NaN
4,NaN,1,남
...,...,...,...
981,NaN,T211,중재이하
982,NaN,T212,고재
983,NaN,T213,대재이상
984,승수(가구)_weight,NaN,NaN


In [8]:
df_2021 = pd.read_csv('2021_코드북_복지사회참여문화와여가소득과소비노동.csv')
df_2023 = pd.read_csv('2023_코드북_복지사회참여문화와여가소득과소비노동.csv')

In [9]:
df_2015["항목명"] = df_2015["항목명"].fillna(method="ffill")
df_2017["항목명"] = df_2017["항목명"].fillna(method="ffill")
df_2019["항목명"] = df_2019["항목명"].fillna(method="ffill")

In [10]:
for i in [2015, 2017, 2019]:
    df = globals()[f"df_{i}"]   # df_2015, df_2017, df_2019 불러오기

    count = df["항목명"].value_counts()
    df_clean = df[~(
        (count[df["항목명"]].values > 1) &   
        (df["코드"].isna() & df["코드의미 및 단위"].isna())
    )]

    # 결과 저장
    globals()[f"df_clean_{i}"] = df_clean

df_clean_2021 = df_2021
df_clean_2023 = df_2023

In [11]:
for i in [2015, 2017, 2019]:
    df_clean = globals()[f"df_clean_{i}"]
    df_clean.to_csv(f"{i}_코드북_항목개별정리.csv", index=False, encoding="utf-8-sig")


In [12]:
cols_2015 = set(df_clean_2015["항목명"])
cols_2017 = set(df_clean_2017["항목명"])
cols_2019 = set(df_clean_2019["항목명"])
cols_2021 = set(df_clean_2021["항목명"])
cols_2023 = set(df_clean_2023["항목명"])

In [13]:
all_items = sorted(cols_2015 | cols_2017 | cols_2019 | cols_2021 | cols_2023)
compare_df = pd.DataFrame({
    "항목명": all_items,
    "2015": [item in cols_2015 for item in all_items],
    "2017": [item in cols_2017 for item in all_items],
    "2019": [item in cols_2019 for item in all_items],
    "2021": [item in cols_2021 for item in all_items],
    "2023": [item in cols_2023 for item in all_items],
})

# 각 행에서 False 개수 세기
compare_df["False_count"] = (~compare_df[["2015", "2017", "2019", "2021", "2023"]]).sum(axis=1)

# 출력 옵션: 열 다 보이도록
# 출력 옵션 늘리기
pd.set_option("display.max_rows", None)      # 행 전부 출력
pd.set_option("display.max_columns", None)  # 열 전부 출력
pd.set_option("display.width", None)        # 줄바꿈 없이 한 줄로 출력
pd.set_option("display.max_colwidth", None) # 셀 안 텍스트 길이 제한 해제

print(compare_df)

                                                               항목명   2015  \
0                                                            가구가중값   True   
1                                                             가구번호   True   
2                                                             가구소득  False   
3                                                           가구소득코드   True   
4                                                           가구원가중값   True   
5                                                            가구원번호   True   
6                                                          가구원일련번호  False   
7                                                           가구일련번호  False   
8                                                          가구주관계코드   True   
9                                                         가구주와의 관계  False   
10                                                    개인적 인간관계 만족도  False   
11                                                    개인적인간관계만족도코드  False   

In [14]:
compare_df.to_csv("코드북_항목명_비교.csv", index=False, encoding="utf-8-sig")

In [19]:
import re

def normalize_column(col):
    # 한글+숫자+영문만 남기고, 나머지는 제거
    col = re.sub(r"[^가-힣A-Za-z0-9()_]", "", col)
    # 소문자로 변환
    col = col.lower()
    # 맨 뒷 글자 '코드' 제거
    col = re.sub(r"코드$", "", col)
    return col

delete_blink = [normalize_column(c) for c in compare_df["항목명"]]


In [20]:
from rapidfuzz import process, fuzz

cols = compare_df["항목명"]
matches = {}
for col in cols:
    match = process.extractOne(col, cols, scorer=fuzz.ratio)
    matches[col] = match

In [25]:
# 매칭 결과 저장
results = []
for col in compare_df["항목명"]:
    match, score, _ = process.extractOne(col, delete_blink, scorer=fuzz.ratio)
    results.append([col, match, f"{score:.4f}"])

# 데이터프레임으로 변환
match_df = pd.DataFrame(results, columns=["원본컬럼(df1)", "매칭컬럼(df2)", "유사도"])

# CSV로 저장
match_df.to_csv("column_matching.csv", index=False, encoding="utf-8-sig")

print("column_matching.csv 파일로 저장 완료!")

column_matching.csv 파일로 저장 완료!


# 2015

# 2017

# 2019
"계층의식" -> "계층의식코드"

# 2021
"기부 여부" -> "기부여부"
"계층의식" -> "계층의식코드"


# 2023